# Phase 3: Baseline Ranking System and Failure@K Definition

Objective: Establish a simple, interpretable baseline ranking system and formally define Failure@K to provide a reference point for the analysis. 

## Phase 3 Goals:
1. Formally define Failure@K
2. Implement simple baseline ranker 
3. Evaluate baseline with P@K and NDCG@K for K={1,3,5,10}
4. Initial failure analysis
5. Output for later phases

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ndcg_score
import warnings
warnings.filterwarnings('ignore')

#Display settings
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',100)
pd.set_option('display.float_format','{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi']=100

#setup paths
PROJECT_ROOT=Path.cwd().parent
DATA_RAW=PROJECT_ROOT/'data'/'raw'
DATA_PROCESSED=PROJECT_ROOT/'data'/'processed'

#Creating outputs directory
PHASE3_OUTPUT=DATA_PROCESSED/'phase3_baseline'
PHASE3_OUTPUT.mkdir(parents=True,exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Phase 3 output directory: {PHASE3_OUTPUT}")

Project root: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding
Phase 3 output directory: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase3_baseline


# Loading Data

In [3]:
#Loading processed training data from Phase1
train_file=DATA_PROCESSED/'fold1_train_processed.csv'
print(f"Loading training data from: {train_file}")
train_df=pd.read_csv(train_file)

#Loading phase 2 feature statistics
feature_stats_file=DATA_PROCESSED/'fold1_feature_statistics.csv'
print(f"Loading feature statistics from: {feature_stats_file}")
feature_stats=pd.read_csv(feature_stats_file, index_col=0)

print(f"\nTraining data loaded: {train_df.shape}")
print(f"Feature statistics loaded: {feature_stats.shape}")

#Identifying feature columns
feature_cols = [col for col in train_df.columns if col.startswith('f') and col[1:].isdigit()]
feature_cols=sorted(feature_cols,key=lambda x: int(x[1:]))
num_features=len(feature_cols)

print(f"\nDataset overview:")
print(f"Features: {num_features}")
print(f"Documents: {len(train_df):,}")
print(f"Queries: {train_df['qid'].nunique():,}")
print(f"Labels: {sorted(train_df['label'].unique())}")

Loading training data from: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\fold1_train_processed.csv
Loading feature statistics from: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\fold1_feature_statistics.csv

Training data loaded: (42158, 49)
Feature statistics loaded: (46, 15)

Dataset overview:
Features: 46
Documents: 42,158
Queries: 1,017
Labels: [np.int64(0), np.int64(1), np.int64(2)]


## Loading Test Data

In [4]:
#We need to load and parse the test data
#Using the same parser from Phase 1

def parse_letor_file(filepath):
    data=[]
    with open(filepath, 'r') as f:
        for line_num, line in enumerate(f,1):
            line=line.strip()
            if not line:
                continue

            try:
                #Splitting comment (docid) from features
                parts=line.split("#")
                feature_part=parts[0].strip()

                #Extracting docid if present
                docid=None
                if len(parts)>1:
                    comment=parts[1].strip()
                    if 'docid' in comment:
                        docid=comment.split('=')[1].strip()

                #Parsing feature part
                tokens=feature_part.split()
                
                #Extracting label
                label=int(tokens[0])

                #Extracting qid
                qid_str=tokens[1]
                qid=int(qid_str.split(':')[1])

                #Extracting features
                features={}
                for token in tokens[2:]:
                    fid, value=token.split(':')
                    features[int(fid)]=float(value)

                data.append({
                    'label':label,
                    'qid':qid,
                    'features':features,
                    'docid':docid
                })

            except Exception as e:
                print(f"Error parsing line {line_num}: {e}")
                continue

    return data

#Loading test data
FOLD1_DIR=DATA_RAW/'MQ2007'/'Fold1'
TEST_FILE=FOLD1_DIR/'test.txt'

if not TEST_FILE.exists():
    print("Test file not found")

print(f"\nLoading test data from: {TEST_FILE}")
print(f"File exists: {TEST_FILE.exists()}")

if TEST_FILE.exists():
    test_data=parse_letor_file(str(TEST_FILE))

    #Converting to dataframe
    records=[]
    for item in test_data:
        record={
            'label':item['label'],
            'qid':item['qid'],
            'docid':item['docid']
        }
        for fid, value in item['features'].items():
            record[f'f{fid}']=value
        records.append(record)

    test_df=pd.DataFrame(records)
    print(f"Test data loaded: {test_df.shape}")
    print(f"Queries: {test_df['qid'].nunique():,}")
    print(f"Documents: {len(test_df):,}")

else:
    print(f"Test file not found. Please verify the path.")
    print(f"Expected location: {FOLD1_DIR}")


Loading test data from: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\raw\MQ2007\Fold1\test.txt
File exists: True
Test data loaded: (13652, 49)
Queries: 336
Documents: 13,652


## Phase 2 findings

In [6]:
print("="*60)
print("Phase 2 Findings Review")
print("="*60)

#Identifyiing zero variance features to exclude
zero_var_features=feature_stats[feature_stats['variance']==0].index.tolist()
print(f"\n1. Zero-variance features (to be excluded)")
print(f"Count:{len(zero_var_features)}")
if zero_var_features:
    print(f"Features: {zero_var_features}")
    print(f"Decision: These will be excluded from baseline")
else:
    print(f"No zero-variance features detected")

#Identifying most discriminative features from Phase 2
#We'll look at the features with high variance and good label separation
non_zero_features=[f for f in feature_cols if f not in zero_var_features]

print(f"\n2. Feature Screening for Baseline")
print(f"Total features: {num_features}")
print(f"Non-zero variance features: {len(non_zero_features)}")

#Computing feature discriminative power (simple mean difference between label 0 and 2)
if len(non_zero_features)>0:
    discriminative_power={}
    for feat in non_zero_features:
        label_0_mean=train_df[train_df['label']==0][feat].mean()
        label_2_mean=train_df[train_df['label']==2][feat].mean()
        discriminative_power[feat]=abs(label_2_mean-label_0_mean)

    #Sorting by discriminative power
    sorted_features=sorted(discriminative_power.items(), key=lambda x: x[1], reverse=True)
    print(f"\nTop 10 most discriminative features (by |mean(label=2)-mean(label=0)|):")
    for feat, power in sorted_features[:10]:
        print(f"{feat}:{power:.4f}")

    most_discriminative_feature=sorted_features[0][0]
    print(f"\nMost discriminative feature: {most_discriminative_feature}")

print(f"\n3. Variance Characteristics")
print(f"Mean feature variance: {feature_stats.loc[non_zero_features, 'variance'].mean():.4f}")
print(f"Mean feature sparsity: {feature_stats.loc[non_zero_features, 'pct_zeros'].mean():.2f}%")
print(f"\nPhase 2 conclusion: Strong within-query variance observed")
print(f"Phase 3 decision: Start with raw features (test normalization later)")

Phase 2 Findings Review

1. Zero-variance features (to be excluded)
Count:5
Features: ['f6', 'f7', 'f8', 'f9', 'f10']
Decision: These will be excluded from baseline

2. Feature Screening for Baseline
Total features: 46
Non-zero variance features: 41

Top 10 most discriminative features (by |mean(label=2)-mean(label=0)|):
f23:0.1803
f39:0.1782
f21:0.1570
f37:0.1526
f40:0.1350
f38:0.1255
f24:0.1252
f22:0.1240
f25:0.1024
f31:0.0971

Most discriminative feature: f23

3. Variance Characteristics
Mean feature variance: 0.0760
Mean feature sparsity: 36.83%

Phase 2 conclusion: Strong within-query variance observed
Phase 3 decision: Start with raw features (test normalization later)


We identified in phase 2 that there are 5 zero-variance features (f6-f10), which are now excluded from modeling because they provide no discriminative signal. 

The discriminative features that we listed are candidates for signal, not some guaranteed predictors.

# Task 1: Formally defining Failure@K

Definition:
- Failure@K = A query where NO relevant document appears in the top-K positions
- Relevance definition (primary): relevant if label >= 1
- Relevance definition (sensitivity): relevant if label = 2 (highly relevant only)

In [9]:
print("="*60)
print("Task 1: Formal definition of Failure@K")
print("="*60)

def is_failure_at_k(ranked_labels, k, relevance_threshold=1):
    """
    Determining if a query is a failure at K.
    Parameters:
    ranked_labels: array-like
        Labels in ranked order (predicted ranking)
    k: int
        Top-K cutoff
    relevance_threshold: int, default=1
        Threshold for relevance:
        -1: relevant if label >= 1 (primary definition)
        -2: relevant if label == 2 (sensitivity analysis)

    REturns:
    is_failure: bool
        True if no relevant document appears in top-K
    """
    if k<=0:
        raise ValueError(f"k must be >= 1, got k={k}")
    #if k is bigger than the list length, python just returns the full list

    if len(ranked_labels)==0:
        return True

    top_k_labels=ranked_labels[:k]

    if relevance_threshold==1:
        #Primary definition: relevant=label>=1
        has_relevant=any(label>=1 for label in top_k_labels)
    elif relevance_threshold==2:
        #Sensitivity definition:relevant: label==2
        has_relevant=any(label==2 for label in top_k_labels)
    else:
        raise ValueError(f"Invalid relevance_threshold: {relevance_threshold}")

    return not has_relevant

#Testing the function
print("\nFunction Definition:")
print("is_failure_at_k(ranked_labels, k, relevance_threshold=1)")
print("\nTest cases:")
test_rankings=[
    ([2,1,0,0],1,1,False),      #Relevant at position 1
    ([0,0,2,1],2,1,True),       #No relevant in top-2
    ([1,0,0,2],3,1,False),      #Relevant at position 1
    ([0,1,0,2],1,1,True),       #No relevant in top-1
    ([1,0,0,0],3,2,True),       #No highly relevant in top-3 (threshold=2)
    ([2,0,1,0],1,2,False),      #Highly relevant at position 1
]

for labels, k, threshold, expected in test_rankings:
    result=is_failure_at_k(labels,k,threshold)
    status="Yes" if result==expected else "No"
    print(f"{status} labels={labels}, K={k}, threshold={threshold} -> Failure={result} (expected {expected})")

print("\nFailure@K definition implemented and tested")

Task 1: Formal definition of Failure@K

Function Definition:
is_failure_at_k(ranked_labels, k, relevance_threshold=1)

Test cases:
Yes labels=[2, 1, 0, 0], K=1, threshold=1 -> Failure=False (expected False)
Yes labels=[0, 0, 2, 1], K=2, threshold=1 -> Failure=True (expected True)
Yes labels=[1, 0, 0, 2], K=3, threshold=1 -> Failure=False (expected False)
Yes labels=[0, 1, 0, 2], K=1, threshold=1 -> Failure=True (expected True)
Yes labels=[1, 0, 0, 0], K=3, threshold=2 -> Failure=True (expected True)
Yes labels=[2, 0, 1, 0], K=1, threshold=2 -> Failure=False (expected False)

Failure@K definition implemented and tested


- Primary relevance: "Did I get something helpful?"
- Sensitivity relevance: "Did I get the **best** thing?"
Using both lets us diagnose where and how our ranking model fails

Relevance labels:
0 -> not relevant
1 -> relevant
2 -> highly relevant

These labels are judgements of usefulness for a query.

Primary relevance: A document is considered relevant if label >= 1 (i.e., label 1 or 2)

Interpretation: This answers the question: "Did the user see anything useful in the top-K results?"
This is user-satisfaction-oriented  definition.
 For example:
 Rank          label          Interpretation
 1               1                useful
 2               0              not useful
 3               0              not useful

 Under primary relevance:
 - label 1 counts as relevant
 - top-3 contains a relevant doc
 - Failure@3 = False(success)

 Even if it's not the best result, the user found something helpful.

This aligns with recall-oriented evaluation

Sensitivity relevance: (label==2 only)
A document is considered relevant only if label==2

LAbel 1 does not count.

Interpretation: "Did the user see a high-quality/ ideal result in the top-K?"

This is more precision-focused and high-bar evaluation.

Example:
Same query:
Rank                     Label
1                          1
2                          0
3                          2

Under sensitivity relevance:
- label 1 does not count
- top-1: no label 2 -> Failure@1=True
- top-3: label 2 appears -> Failure@3=False

So, Failure@1 is higher
    Failure@3 improves

This is called "sensitivity" because:
- results are sensitive to ranking quality
- small ranking errors (putting 2 at rank 3 instead of 1) are exposed
- the evaluation becomes more demanding

Side-by-side comparison:
Rank                 Label
1                      1
2                      1
3                      0
4                      2

At K=2
 - Primary relevance (>=1):
     * labels[1,1] -> success
     * Failure@2 = False

 - Sensitivity relevance (=2):
     * no label 2 in top-2
     * Failure@2 = True

Same ranking, different interpretation

This tells us:
The model finds acceptable results early, but struggles to surface the best result early. 

# Task 2: Implementing Simple Baseline Ranker

### Baseline Choice: Pointwise Logistic Regression

Justification:
1. Simplicity: Linear model, easy to interpret
2. Transparency: Coefficients show feature importance
3. Established: Standard baseline
4. Multi-class support: Can handle labels {0,1,2}
5. No tuning needed: Default parameters for baseline

Implementation details:
- Using raw features (no normalization)
- Excluding zero-variance features
- Predicting probability scores, rank by score
- No hyperparameter tuning (default sklearn parameters)


In [12]:
print("="*60)
print("Task 2: Baseline Ranker Implementation")
print("="*80)

print("\nBaseline Choice: Pointwise Logistic Regression")
print("\nJustification")
print("1. Simple, interpretable linear model")
print("2. Transparent feature weights")
print("3. Standard LTR baseline")
print("4. Handles multi-class labels {0,1,2}")
print("5. No hyperparameter tuning needed")

#Preparing features (excluding zero-variance)
if len(zero_var_features)>0:
    active_features=[f for f in feature_cols if f not in zero_var_features]
    print(f"\nFeature preparation:")
    print(f" Total features: {len(feature_cols)}")
    print(f" Zero-variance excluded: {len(zero_var_features)}")
    print(f" Active features: {len(active_features)}")
else:
    active_features=feature_cols
    print(f"\nFeature preparation")
    print(f" All {len(feature_cols)} features active (no zero-variance features)")

#Extracting training data
X_train=train_df[active_features].values
y_train=train_df['label'].values

print(f"\nTraining data:")
print(f" X_train shape: {X_train.shape}")
print(f" y_train shape: {y_train.shape}")
print(f" Label distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")

#Training baseline model
print(f"\n Training Logistic Regression")
baseline_model=LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000,
    random_state=36
)

baseline_model.fit(X_train, y_train)
print(f"Model trained successfully")

#Extracting test data
if 'test_df' in locals():
    X_test=test_df[active_features].values
    y_test=test_df['label'].values
    qids_test=test_df['qid'].values

    print(f"\nTest data:")
    print(f" X_test shape: {X_test.shape}")
    print(f" y_test shape: {y_test.shape}")
    print(f" Unique queries: {len(np.unique(qids_test))}")
else:
    print(f"\nTest data not loaded, cannot proceed with evaluation")

Task 2: Baseline Ranker Implementation

Baseline Choice: Pointwise Logistic Regression

Justification
1. Simple, interpretable linear model
2. Transparent feature weights
3. Standard LTR baseline
4. Handles multi-class labels {0,1,2}
5. No hyperparameter tuning needed

Feature preparation:
 Total features: 46
 Zero-variance excluded: 5
 Active features: 41

Training data:
 X_train shape: (42158, 41)
 y_train shape: (42158,)
 Label distribution: {np.int64(0): np.int64(31585), np.int64(1): np.int64(8388), np.int64(2): np.int64(2185)}

 Training Logistic Regression
Model trained successfully

Test data:
 X_test shape: (13652, 41)
 y_test shape: (13652,)
 Unique queries: 336


## Generating Predictions and Rankings

In [15]:
if 'test_df' in locals():
    print("Generating predictions for test set")

    #Predicting probability scores (using probability of highest class as score)
    #This gives us a continuous score for ranking
    y_pred_proba=baseline_model.predict_proba(X_test)

    #Using weighted score: sum of (class_value*probability)
    #THis gives higher scores to documents predicted to be more relevant
    classes=baseline_model.classes_
    scores=np.sum(y_pred_proba*classes.reshape(1,-1),axis=1)

    #adding scores to test dataframe
    test_df['score']=scores

    print(f"Scores computed")
    print(f"\nScore statistics:")
    print(f"Min: {scores.min():.4f}")
    print(f"Max: {scores.max():.4f}")
    print(f"Mean: {scores.mean():.4f}")
    print(f"Std: {scores.std():.4f}")

    #Creating rankings per query
    print(f"\nCreating per-query rankings")
    test_df['rank']=test_df.groupby('qid')['score'].rank(ascending=False,method='first').astype(int)

    print(f"Rankings created")
    print(f"\nSample query (qid={test_df['qid'].iloc[0]}):")
    sample_qid=test_df['qid'].iloc[0]
    sample_query=test_df[test_df['qid']==sample_qid][['qid','label','score','rank']].sort_values('rank')
    display(sample_query.head(10))

else:
    print("Test data not available")

Generating predictions for test set
Scores computed

Score statistics:
Min: 0.0168
Max: 1.1613
Mean: 0.3104
Std: 0.1693

Creating per-query rankings
Rankings created

Sample query (qid=7968):


,qid,label,score,rank
17,7968,2,0.6277,1
25,7968,1,0.5291,2
6,7968,1,0.4723,3
23,7968,1,0.4663,4
12,7968,0,0.4160,5
4,7968,0,0.4021,6
31,7968,1,0.3945,7
14,7968,1,0.3919,8
15,7968,1,0.3879,9
1,7968,1,0.3808,10
